# 04 - Environmental Encoding and Plant Clustering

## Role of This Notebook
Here, `house_plants.csv` is translated into a stable environmental table, and species are then grouped by similarity of requirements. The goal is no longer only to normalize text, but to build a reproducible bridge between the botanical catalog and the spatial dataset.

## Main Criterion
1. Convert textual descriptions into interpretable environmental proxies.
2. Test only `k=6` and `k=7`, because we want a small number of operational groups.
3. Export one encoded table by species and another table by cluster profile so notebooks 05-07 can work from a consistent base.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.edgecolor': '#777777',
    'axes.grid': True,
    'grid.color': '#e6e6e6',
    'grid.linestyle': '-',
    'grid.linewidth': 0.8,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 10,
})

ROOT = Path.cwd().resolve().parent
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
plants = pd.read_csv(RAW_DIR / 'house_plants.csv')
plants.head()

## 1. Translating Botanical Text Into Environmental Proxies
The 01-07 series needs variables that can be read by rules and by ML. For that reason, this stage preserves the interpretability of the original notebook 04, but takes it closer to the reference notebook `gb_plantgroup_jnotebook_v2.ipynb`.

The idea is not to reconstruct the full real horticulture of a species, but to obtain a sufficiently consistent profile in light, humidity, watering, and temperature so plants can be grouped and compared.


In [ ]:
for col in ['latin', 'family', 'category', 'origin', 'climate', 'common', 'ideallight', 'toleratedlight', 'watering', 'use']:
    plants[col] = plants[col].fillna('N/A').astype(str).str.strip()

def normalize_light_need(row):
    ideal = row['ideallight'].lower()
    tolerated = row['toleratedlight'].lower()
    category = row['category'].lower()

    if '6 or more hours' in ideal or 'direct sunlight' in ideal or 'direct sunlight' in tolerated:
        return 'high_light'
    if any(keyword in category for keyword in ['fern', 'aglaonema']) and 'diffused' in tolerated:
        return 'shade_tolerant'
    if 'bright light' in ideal and 'diffused' in tolerated:
        return 'bright_indirect'
    if 'bright light' in ideal:
        return 'medium_indirect'
    return 'medium_indirect'

def supports_direct_sun(text):
    text = str(text).lower()
    return int('direct sunlight' in text or '6 or more hours' in text)

def supports_diffused_light(text):
    return int('diffused' in str(text).lower())

def build_use_score(text):
    text = str(text).lower()
    score = 0
    for keyword in ['potted plant', 'table top', 'ground cover']:
        if keyword in text:
            score += 1
    return min(score, 3)

def build_light_flexibility(row):
    return (
        supports_direct_sun(row['ideallight']) +
        supports_direct_sun(row['toleratedlight']) +
        supports_diffused_light(row['toleratedlight'])
    )

def infer_light_windows(light_need):
    if light_need == 'high_light':
        return pd.Series([6.0, 8.0, 0.0, 2.0, 10000.0, 100000.0])
    if light_need == 'shade_tolerant':
        return pd.Series([0.0, 1.0, 7.0, 9.0, 500.0, 8000.0])
    if light_need == 'bright_indirect':
        return pd.Series([0.0, 2.0, 6.0, 8.0, 800.0, 10000.0])
    return pd.Series([1.0, 4.0, 4.0, 7.0, 1500.0, 30000.0])

def infer_humidity_range(row):
    climate = row['climate'].lower()
    category = row['category'].lower()
    if 'fern' in category or 'humid' in climate:
        return pd.Series([60.0, 90.0])
    if 'arid' in climate or 'cactus' in category or 'succulent' in category:
        return pd.Series([20.0, 50.0])
    if 'subtropical' in climate:
        return pd.Series([40.0, 70.0])
    return pd.Series([45.0, 75.0])

def infer_watering_scores(text):
    text = str(text).lower()
    if 'must not be dry' in text:
        return pd.Series([4.0, 0.10])
    if 'keep moist between watering' in text:
        return pd.Series([3.5, 0.25])
    if 'half dry' in text:
        return pd.Series([3.0, 0.45])
    if 'must be dry between watering' in text or 'water only when the soil is dry' in text:
        return pd.Series([1.0, 0.90])
    return pd.Series([2.0, 0.60])

plants['normalized_light_need'] = plants.apply(normalize_light_need, axis=1)
plants[['direct_sun_min_h_day', 'direct_sun_max_h_day', 'preferred_shadow_min_h_day', 'preferred_shadow_max_h_day', 'lux_min', 'lux_max']] = plants['normalized_light_need'].apply(infer_light_windows)
plants[['humidity_min_percent', 'humidity_max_percent']] = plants.apply(infer_humidity_range, axis=1)
plants[['soil_moisture_class_0_4', 'drought_tolerance_0_1']] = plants['watering'].apply(infer_watering_scores)
plants['supports_direct_sun'] = plants.apply(lambda row: max(supports_direct_sun(row['ideallight']), supports_direct_sun(row['toleratedlight'])), axis=1)
plants['supports_diffused_light'] = plants['toleratedlight'].apply(supports_diffused_light)
plants['supports_bright_light'] = plants['ideallight'].str.lower().str.contains('bright light').astype(int)
plants['light_flexibility'] = plants.apply(build_light_flexibility, axis=1)
plants['indoor_use_score'] = plants['use'].apply(build_use_score)
plants['temp_context_score'] = np.where((plants['tempmin_celsius'] <= 18) & (plants['tempmax_celsius'] >= 28), 2, 1)
plants['temp_preferred_min_c'] = plants['tempmin_celsius']
plants['temp_preferred_max_c'] = plants['tempmax_celsius']
plants['plant_group'] = plants['normalized_light_need']

plants[['latin', 'normalized_light_need', 'direct_sun_min_h_day', 'direct_sun_max_h_day', 'lux_min', 'lux_max', 'humidity_min_percent', 'humidity_max_percent']].head()

## 2. Visual Reading of the Derived Variables
Before grouping species, it is useful to inspect the distribution of the environmental variables. This helps verify that the encoding did not create absurdly uniform profiles and that there is enough variety to justify clustering.


In [ ]:
plants['sun_center_h_day'] = (plants['direct_sun_min_h_day'] + plants['direct_sun_max_h_day']) / 2
plants['sun_range_h_day'] = plants['direct_sun_max_h_day'] - plants['direct_sun_min_h_day']
plants['shadow_center_h_day'] = (plants['preferred_shadow_min_h_day'] + plants['preferred_shadow_max_h_day']) / 2
plants['shadow_range_h_day'] = plants['preferred_shadow_max_h_day'] - plants['preferred_shadow_min_h_day']
plants['lux_center'] = (plants['lux_min'] + plants['lux_max']) / 2
plants['lux_range'] = plants['lux_max'] - plants['lux_min']
plants['humidity_center_percent'] = (plants['humidity_min_percent'] + plants['humidity_max_percent']) / 2
plants['humidity_range_percent'] = plants['humidity_max_percent'] - plants['humidity_min_percent']
plants['temp_preferred_center_c'] = (plants['temp_preferred_min_c'] + plants['temp_preferred_max_c']) / 2
plants['temp_preferred_range_c'] = plants['temp_preferred_max_c'] - plants['temp_preferred_min_c']

visual_features = [
    'sun_center_h_day', 'lux_center', 'humidity_center_percent',
    'soil_moisture_class_0_4', 'drought_tolerance_0_1', 'temp_preferred_center_c'
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
feature_colors = ['#4e79a7', '#f28e2b', '#59a14f', '#e15759', '#b07aa1', '#76b7b2']

for ax, feature, color in zip(axes, visual_features, feature_colors):
    ax.hist(plants[feature].dropna(), bins=20, color=color, edgecolor='#444444', linewidth=0.6, alpha=0.85)
    ax.set_title(feature)
    ax.set_xlabel(feature)
    ax.set_ylabel('Count')

plt.tight_layout()
plt.show()

### Conclusion From This Visual Reading
The distribution of the variables confirms that the encoding preserves enough environmental diversity to justify clustering. There is no single absolute dominant profile: high-light species, low diffuse-light groups, and an intermediate block coexist. This variety explains why the project does not remain with a single manual light taxonomy and instead moves toward a richer cluster structure.


## 3. Building Features for Clustering
The derived variables condense the most important part of the botanical language for the project: how much light a plant requires, how much humidity it tolerates, and how rigid its care is. With this, we can group species by approximate environmental behavior.


In [ ]:
cluster_cols = [
    'sun_center_h_day', 'sun_range_h_day', 'shadow_center_h_day', 'shadow_range_h_day',
    'lux_center', 'lux_range', 'humidity_center_percent', 'humidity_range_percent',
    'soil_moisture_class_0_4', 'drought_tolerance_0_1', 'temp_preferred_center_c', 'temp_preferred_range_c'
]

X = plants[cluster_cols].fillna(plants[cluster_cols].median(numeric_only=True))
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

cluster_results = []
for k in [6, 7]:
    model = KMeans(n_clusters=k, random_state=42, n_init=50)
    labels = model.fit_predict(X_scaled)
    cluster_results.append({
        'k': k,
        'inertia': model.inertia_,
        'silhouette_score': silhouette_score(X_scaled, labels)
    })

cluster_results = pd.DataFrame(cluster_results).sort_values('k')
display(cluster_results)
selected_k = int(cluster_results.sort_values('silhouette_score', ascending=False).iloc[0]['k'])
print(f'Selected k: {selected_k}')

## 4. Selecting the Number of Clusters
Here, `inertia` and `silhouette_score` are visualized. Since only `k=6` and `k=7` are compared, the chart does not work as a broad classic elbow plot, but it does provide clear evidence for why the final cluster count is chosen.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(cluster_results['k'], cluster_results['inertia'], marker='o', color='#4e79a7', linewidth=2)
axes[0].axvline(selected_k, color='#e15759', linestyle='--', linewidth=1.5)
axes[0].set_title('Inertia by k')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Inertia')

axes[1].plot(cluster_results['k'], cluster_results['silhouette_score'], marker='o', color='#59a14f', linewidth=2)
axes[1].axvline(selected_k, color='#e15759', linestyle='--', linewidth=1.5)
axes[1].set_title('Silhouette score by k')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Silhouette score')

plt.tight_layout()
plt.show()

### Conclusion About k Selection
Between the two compared options, `k=7` provides better separation without fragmenting the set too much. In the current results, inertia drops from `398.16` to `348.41`, and the `silhouette_score` rises from `0.4855` to `0.5221`. For that reason, the pipeline keeps seven botanical clusters before consolidating them into a more compact operational layer.


## 5. Fitting the Final Model and Visually Reading the Clusters
The final cluster is not interpreted as an absolute botanical taxonomy. It is interpreted as an operational group of species with similar environmental requirements. This is the unit that later communicates with the spatial dataset.


In [ ]:
kmeans = KMeans(n_clusters=selected_k, random_state=42, n_init=50)
plants['plant_variety_cluster'] = kmeans.fit_predict(X_scaled)
plants['plant_cluster_label'] = 'cluster_' + plants['plant_variety_cluster'].astype(str)

cluster_profiles = (
    plants.groupby(['plant_variety_cluster', 'plant_cluster_label'])
    .agg(
        plant_count=('id', 'count'),
        dominant_categories=('category', lambda s: '; '.join(s.value_counts().head(4).index.astype(str))),
        dominant_climates=('climate', lambda s: '; '.join(s.value_counts().head(4).index.astype(str))),
        example_species=('latin', lambda s: '; '.join(s.head(4).astype(str))),
        sun_min_h_day=('direct_sun_min_h_day', 'mean'),
        sun_max_h_day=('direct_sun_max_h_day', 'mean'),
        lux_min=('lux_min', 'mean'),
        lux_max=('lux_max', 'mean'),
        humidity_min_percent=('humidity_min_percent', 'mean'),
        humidity_max_percent=('humidity_max_percent', 'mean'),
        temp_preferred_center_c=('temp_preferred_center_c', 'mean')
    )
    .reset_index()
)

cluster_profiles['sun_center_h_day'] = (cluster_profiles['sun_min_h_day'] + cluster_profiles['sun_max_h_day']) / 2
cluster_profiles['lux_center'] = (cluster_profiles['lux_min'] + cluster_profiles['lux_max']) / 2
cluster_profiles['humidity_center_percent'] = (cluster_profiles['humidity_min_percent'] + cluster_profiles['humidity_max_percent']) / 2

lux_min_global = cluster_profiles['lux_center'].min()
lux_max_global = cluster_profiles['lux_center'].max()
if lux_max_global == lux_min_global:
    cluster_profiles['lux_norm'] = 0.5
else:
    cluster_profiles['lux_norm'] = (cluster_profiles['lux_center'] - lux_min_global) / (lux_max_global - lux_min_global)

cluster_profiles['sun_norm_req'] = (cluster_profiles['sun_center_h_day'] / 12.0).clip(0, 1)
cluster_profiles['plant_light_index'] = 0.5 * cluster_profiles['sun_norm_req'] + 0.5 * cluster_profiles['lux_norm']

bridge_key_cols = ['sun_min_h_day', 'sun_max_h_day', 'lux_min', 'lux_max', 'sun_center_h_day', 'lux_center', 'plant_light_index']
bridge_profiles = cluster_profiles[bridge_key_cols].drop_duplicates().reset_index(drop=True).copy()
bridge_profiles['bridge_cluster_id'] = bridge_profiles.index.astype(int)
bridge_profiles['bridge_cluster_label'] = 'bridge_cluster_' + bridge_profiles['bridge_cluster_id'].astype(str)

cluster_profiles = cluster_profiles.merge(bridge_profiles, on=bridge_key_cols, how='left')
cluster_profiles['source_plant_clusters'] = cluster_profiles['plant_cluster_label']

plants = plants.merge(
    cluster_profiles[['plant_variety_cluster', 'plant_light_index', 'bridge_cluster_id', 'bridge_cluster_label']],
    on='plant_variety_cluster',
    how='left'
)

display(cluster_profiles)
display(bridge_profiles)

In [ ]:
cluster_counts = plants['plant_cluster_label'].value_counts().sort_index()
cluster_cmap = plt.get_cmap('tab10')
cluster_colors = {label: cluster_cmap(i % 10) for i, label in enumerate(cluster_counts.index)}

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

bars = axes[0].bar(
    cluster_counts.index,
    cluster_counts.values,
    color=[cluster_colors[label] for label in cluster_counts.index],
    edgecolor='#444444',
    linewidth=0.6
)
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height(), str(int(bar.get_height())), ha='center', va='bottom', fontsize=9)
axes[0].set_title('Species per plant cluster')
axes[0].set_xlabel('Plant cluster')
axes[0].set_ylabel('Number of species')
axes[0].tick_params(axis='x', rotation=45)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
centroids_pca = pca.transform(kmeans.cluster_centers_)

for cluster_id in sorted(plants['plant_variety_cluster'].unique()):
    subset = plants[plants['plant_variety_cluster'] == cluster_id]
    label = f'cluster_{cluster_id}'
    mask = plants['plant_variety_cluster'] == cluster_id
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], s=40, alpha=0.75, color=cluster_colors[label], edgecolor='#333333', linewidth=0.3, label=label)

for cluster_id, centroid in enumerate(centroids_pca):
    label = f'cluster_{cluster_id}'
    axes[1].scatter(centroid[0], centroid[1], s=220, marker='X', color=cluster_colors[label], edgecolor='#111111', linewidth=1.5)
    axes[1].text(centroid[0], centroid[1], f'  C{cluster_id}', fontsize=10, weight='bold')

axes[1].set_title('PCA view of plant clusters')
axes[1].set_xlabel('PCA 1')
axes[1].set_ylabel('PCA 2')
axes[1].legend(frameon=False, bbox_to_anchor=(1.02, 1), loc='upper left')

plt.tight_layout()
plt.show()

### What These Clustering Visualizations Contribute
- The bar chart by cluster shows whether any group is overrepresented.
- PCA does not prove by itself that the clustering is perfect, but it does help show whether there is reasonable separation between groups.

If the clouds partially overlap, the correct reading is not that the method fails, but that there are species with intermediate profiles between nearby clusters. In addition, when two botanical clusters share the same useful profile for the spatial bridge, they are later consolidated into the same `bridge_cluster`.


In [ ]:
profile_plot = cluster_profiles[['plant_cluster_label', 'sun_center_h_day', 'lux_center', 'humidity_center_percent']].copy()
profile_plot = profile_plot.sort_values('plant_cluster_label')

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
summary_cols = ['sun_center_h_day', 'lux_center', 'humidity_center_percent']
summary_titles = ['Mean daily sun center', 'Mean lux center', 'Mean humidity center']
summary_colors = ['#4e79a7', '#f28e2b', '#59a14f']

for ax, col, title, color in zip(axes, summary_cols, summary_titles, summary_colors):
    ax.bar(profile_plot['plant_cluster_label'], profile_plot[col], color=color, edgecolor='#444444', linewidth=0.6)
    ax.set_title(title)
    ax.set_xlabel('Plant cluster')
    ax.set_ylabel(col)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### Conclusion About the Obtained Profiles
The cluster summary shows three strong patterns. First, clusters `0` and `2` concentrate high-light species with `sun_center_h_day = 7.0` and `lux_center = 55000`. Second, clusters `1` and `4` represent low diffuse-light profiles with `sun_center_h_day = 1.0` and `lux_center = 5400`. Third, clusters `5` and `6` remain in even stricter shade. This partial repetition explains why the pipeline later introduces `bridge clusters`: several botanical differences are real within the catalog, but not necessarily distinguishable from the available geometry.

They also reveal an important point in the pipeline: several botanical clusters can be different inside the catalog, but practically equivalent for the geometric crosswalk. For that reason, the project later uses a more compact `bridge cluster` layer.


## 6. Notebook Outputs
Five artifacts are exported:

1. `plants_encoded.csv`: table by species with derived features and assigned cluster.
2. `plants_cluster_profiles.csv`: average environmental profile by cluster.
3. `plant_cluster_model_selection.csv`: comparison between `k=6` and `k=7`.
4. `plant_cluster_distribution.csv`: species count by cluster.
5. `bridge_cluster_profiles.csv`: unique operational profiles for the bridge with geometry.

These outputs allow notebook 05 to build bridge labels between plants and geometry without having to reinterpret the botanical CSV again.


In [ ]:
encoded_columns = [
    'id', 'latin', 'family', 'category', 'origin', 'climate', 'common',
    'tempmax_celsius', 'tempmin_celsius', 'ideallight', 'toleratedlight', 'watering', 'use',
    'normalized_light_need', 'supports_direct_sun', 'supports_diffused_light', 'supports_bright_light',
    'light_flexibility', 'indoor_use_score', 'temp_context_score', 'plant_group',
    'direct_sun_min_h_day', 'direct_sun_max_h_day', 'preferred_shadow_min_h_day', 'preferred_shadow_max_h_day',
    'lux_min', 'lux_max', 'humidity_min_percent', 'humidity_max_percent',
    'soil_moisture_class_0_4', 'drought_tolerance_0_1',
    'sun_center_h_day', 'sun_range_h_day', 'shadow_center_h_day', 'shadow_range_h_day',
    'lux_center', 'lux_range', 'humidity_center_percent', 'humidity_range_percent',
    'temp_preferred_min_c', 'temp_preferred_max_c', 'temp_preferred_center_c', 'temp_preferred_range_c',
    'plant_variety_cluster', 'plant_cluster_label', 'plant_light_index', 'bridge_cluster_id', 'bridge_cluster_label'
]

cluster_distribution = plants['plant_cluster_label'].value_counts().rename_axis('plant_cluster_label').reset_index(name='count')

plants[encoded_columns].to_csv(PROCESSED_DIR / 'plants_encoded.csv', index=False)
cluster_profiles.to_csv(PROCESSED_DIR / 'plants_cluster_profiles.csv', index=False)
bridge_profiles.to_csv(PROCESSED_DIR / 'bridge_cluster_profiles.csv', index=False)
cluster_results.to_csv(PROCESSED_DIR / 'plant_cluster_model_selection.csv', index=False)
cluster_distribution.to_csv(PROCESSED_DIR / 'plant_cluster_distribution.csv', index=False)

print('Saved:', PROCESSED_DIR / 'plants_encoded.csv')
print('Saved:', PROCESSED_DIR / 'plants_cluster_profiles.csv')
print('Saved:', PROCESSED_DIR / 'bridge_cluster_profiles.csv')
print('Saved:', PROCESSED_DIR / 'plant_cluster_model_selection.csv')
print('Saved:', PROCESSED_DIR / 'plant_cluster_distribution.csv')